In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import json
import os
import csv
import torch
import random
from transformers import set_seed

# Fijar semilla
SEED = 42
set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Cargar dataset

In [2]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from datasets import load_dataset

path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
dataset_anx = load_dataset("csv", data_files={
    "train": path + "anxiety_train.csv",
    "test": path + "anxiety_test.csv",
    "validation": path + "anxiety_val.csv"
})

dataset_dep = load_dataset("csv", data_files={
    "train": path + "depression_train.csv",
    "test": path + "depression_test.csv",
    "validation": path + "depression_val.csv"
})

dataset_ed = load_dataset("csv", data_files={
    "train": path + "ed_train.csv",
    "test": path + "ed_test.csv",
    "validation": path + "ed_val.csv"
})

dataset_multiclass = load_dataset("csv", data_files={
    "train": path + "multiclass_train.csv",
    "test": path + "multiclass_test.csv",
    "validation": path + "multiclass_val.csv"
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

## Modelos

**Configuración específica por modelo.**

Los valores se han elegido de forma práctica teniendo en cuenta:
- el tamaño del modelo (número de parámetros)
- la complejidad de la arquitectura
- la longitud máxima de los textos (max_length)
- las limitaciones de memoria de Colab (GPU T4)

Modelos más ligeros (DistilBETO) permiten batches (train_bs y eval_bs) mayores.

Modelos más grandes o complejos (RoBERTa-BNE, mDeBERTa) requieren batches más pequeños.

BigBird está pensado para secuencias largas, por lo que se prioriza aumentar max_length y se reduce el batch size para evitar errores de memoria.

En evaluación se usa un batch mayor que en entrenamiento porque no se calculan gradientes
y el consumo de memoria es menor.

In [132]:
MODELS = {
    "distilbeto": "dccuchile/distilbert-base-spanish-uncased",
    "roberta_bne": "illuin/roberta-large-bne",
    "mdeberta": "microsoft/mdeberta-v3-base",
    "bigbird": "google/bigbird-roberta-base"
}

MODEL_CONFIGS = {
    "distilbeto": {
        "max_length": 512,
        "train_bs": 16,
        "eval_bs": 64
    },
    "roberta_bne": {
        "max_length": 512,
        "train_bs": 8,
        "eval_bs": 32
    },
    "mdeberta": {
        "max_length": 512,
        "train_bs": 8,
        "eval_bs": 32
    },
    "bigbird": {
        "max_length": 1536,  # Entre 1024–4096
        "train_bs": 1,
        "eval_bs": 2
    }
}

## Seleccionar dataset y modelo

In [133]:
# Seleccionar dataset: dataset_anx | dataset_dep | dataset_ed | dataset_multiclass
dataset = dataset_anx # <-----

# Seleccionar modelo: distilbeto | roberta_bne | mdeberta | bigbird
model_key = "bigbird" # <-----
model_name = MODELS[model_key]
config = MODEL_CONFIGS[model_key]

# Generar path de salida para nombrar CSV, zip...
if dataset is dataset_anx:
    dataset_name = "anxiety"
    num_clases = 2
elif dataset is dataset_dep:
    dataset_name = "depression"
    num_clases = 2
elif dataset is dataset_ed:
    dataset_name = "ed"
    num_clases = 2
else:
    dataset_name = "multiclass"
    num_clases = 4
model_id = model_name.split("/")[-1]
path_salida = f"{model_id}_{dataset_name}"

## Cargar Modelo

En esta sección se carga el modelo preentrenado seleccionado junto con su tokenizador correspondiente.

El modelo se inicializa para una tarea de clasificación de textos, ajustando el número de clases según el tipo de problema (binario o multiclase).

In [134]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_clases
)

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at google/bigbird-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenizar el dataset

Aquí se transforman los textos originales en representaciones numéricas que el modelo puede procesar.

La tokenización convierte cada texto en secuencias de tokens, aplicando truncado o padding hasta una longitud máxima definida según el modelo utilizado.

In [135]:
def tokenize(batch):
    return tokenizer(
        batch["texto"],
        padding="max_length",
        truncation=True,
        max_length=config["max_length"]
    )

encoded_dataset = dataset.map(tokenize, batched=True)
if num_clases == 2:
  # Clasificación binaria
  encoded_dataset = encoded_dataset.rename_column("bs", "labels")
  encoded_dataset = encoded_dataset.remove_columns(["texto", "user_id"])
else:
  # Clasificación multiclase
  encoded_dataset = encoded_dataset.rename_column("label_mc", "labels")
  encoded_dataset = encoded_dataset.remove_columns(["texto", "user_id", "bs", "trastorno"])

encoded_dataset.set_format("torch")

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

## Calcular los pesos de las clases

En esta parte se calculan pesos para cada clase a partir de su frecuencia en el conjunto de entrenamiento.

Estos pesos se utilizan posteriormente en la función de pérdida para mitigar el efecto del desbalance entre clases y mejorar el rendimiento en las clases minoritarias.

In [136]:
from collections import Counter

# Extraer labels como lista de Python
labels = [int(x) for x in encoded_dataset["train"]["labels"]]

# Contar frecuencia por clase
class_counts = Counter(labels)
print("Frecuencia de clases:", class_counts)

num_classes = len(class_counts)

# Pesos inversamente proporcionales a la frecuencia
class_weights = np.array(
    [1.0 / class_counts[i] for i in range(num_classes)],
    dtype=np.float32
)

# Normalizar (opcional pero recomendable)
class_weights = class_weights / class_weights.sum() * num_classes

class_weights = torch.tensor(class_weights)
print("Pesos de clase:", class_weights)

Frecuencia de clases: Counter({1: 310, 0: 40})
Pesos de clase: tensor([1.7714, 0.2286])


## Fine-tuning del modelo

En esta sección se realiza el entrenamiento del modelo sobre el conjunto de datos específico.

El modelo preentrenado se ajusta a la tarea concreta mediante fine-tuning, utilizando una función de pérdida ponderada, métricas de evaluación y un mecanismo de early stopping para evitar sobreajuste.

In [137]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import torch.nn as nn

# Crear un Trainer personalizado con loss ponderada
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Definir loss ponderada
        loss_func = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_func(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Configurar el entrenamiento
training_args = TrainingArguments(
    seed=42,
    data_seed=42,

    output_dir = path_salida,            # Carpeta donde se guardarán los modelos y resultados
    eval_strategy="epoch",               # Evaluar al final de cada época
    save_strategy="epoch",               # Guardar un checkpoint al final de cada época
    load_best_model_at_end=True,         # Cargar automáticamente el mejor modelo al finalizar el entrenamiento
    metric_for_best_model="f1",          # Usar la métrica F1 para decidir cuál es el mejor modelo
    greater_is_better=True,              # Un valor más alto de F1 indica un mejor modelo

    num_train_epochs=10,                            # Número total de épocas de entrenamiento
    per_device_train_batch_size=config["train_bs"], # Tamaño de batch para entrenamiento
    per_device_eval_batch_size=config["eval_bs"],    # Tamaño de batch para evaluación (puede ser mayor)

    warmup_ratio = 0.1,                  # Porcentaje del entrenamiento dedicado al calentamiento para estabilizar la tasa de aprendizaje
    weight_decay=0.01,                   # Regularización L2 para evitar sobreajuste

    save_total_limit=1,                  # Solo conservar 1 checkpoint (ahorra espacio)
    fp16=True,                           # Entrenamiento en precisión mixta (más rápido y menor uso de VRAM)

    report_to="none"
)

# Definir métricas de evaluación
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }

# Entrenar el modelo
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 6251 has 14.73 GiB memory in use. Of the allocated memory 14.55 GiB is allocated by PyTorch, and 44.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Evaluar en el conjunto de test

In [112]:
# Obtener los logits, labels reales y métricas
predictions_output = trainer.predict(encoded_dataset["test"])

# Logits -> predicciones
logits = predictions_output.predictions
preds = np.argmax(logits, axis=1)

# Añadir predicciones al dataset original de test
test_df = dataset["test"].to_pandas()
test_df["pred"] = preds

# Almacenar predicciones en CSV
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{path_salida}.csv"
)
test_df.to_csv(output_path, index=False)

### Comprobación de predicciones

In [113]:
# Etiquetas verdaderas
true_labels = [int(x) for x in encoded_dataset["test"]["labels"]]

f1_test = f1_score(true_labels, preds, average="macro")

# F1 ponderada (tiene en cuenta el desbalance)
f1_weighted = f1_score(true_labels, preds, average="weighted")

# F1 micro (equivalente a accuracy en multiclase)
f1_micro = f1_score(true_labels, preds, average="micro")

print("F1 macro:", f1_test)
print("F1 weighted:", f1_weighted)
print("F1 micro:", f1_micro)

F1 macro: 0.7664610829699734
F1 weighted: 0.7654936311956735
F1 micro: 0.7661691542288557


## Guardar el modelo final

In [ ]:
from google.colab import files
!zip -r {path_salida}.zip {path_salida}
files.download(f"{path_salida}.zip")

  adding: distilbert-base-spanish-uncased_ed/ (stored 0%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/ (stored 0%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/model.safetensors (deflated 7%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/config.json (deflated 49%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/training_args.bin (deflated 53%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/rng_state.pth (deflated 26%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/optimizer.pt (deflated 32%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/trainer_state.json (deflated 65%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/scaler.pt (deflated 64%)
  adding: distilbert-base-spanish-uncased_ed/checkpoint-45/scheduler.pt (deflated 62%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>